# Notebook 03 — Classification: Arrest Prediction (Task 1)

Trains and evaluates Logistic Regression, Random Forest, and SVM on the feature-engineered dataset.

**Prerequisites:** `02_feature_engineering.ipynb` must have saved `data/processed/chicago_features.csv`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.features import get_feature_matrix
from src.models.classification import (
    get_models, train_evaluate_all, cross_validate_model, tune_random_forest
)
from src.evaluation.metrics import (
    plot_confusion_matrices, plot_roc_curves,
    plot_metrics_comparison, plot_feature_importance, save_metrics_csv
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../outputs/reports', exist_ok=True)

## 1. Load feature-engineered data

In [ ]:
df = pd.read_csv('../data/processed/chicago_features.csv', low_memory=False)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
X, y, feature_names = get_feature_matrix(df)
print(f'X: {X.shape}  |  y: {y.shape}  |  Arrest rate: {y.mean():.1%}')
print(f'Features ({len(feature_names)}): {feature_names}')

## 2. Train and evaluate all classifiers

In [ ]:
results, X_test, y_test = train_evaluate_all(
    X, y, output_dir='../outputs/models'
)

## 3. Performance comparison table

In [ ]:
rows = []
for name, m in results.items():
    row = {k: v for k, v in m.items() if k not in ('model',)}
    row['Model'] = name
    rows.append(row)
df_metrics = pd.DataFrame(rows).set_index('Model')
print(df_metrics.to_string())

save_metrics_csv(results, path='../outputs/reports/classification_results.csv')

## 4. Visualisations

In [ ]:
fitted_models = {name: m['model'] for name, m in results.items()}

plot_metrics_comparison(results, output_dir='../outputs/figures')

In [ ]:
plot_confusion_matrices(fitted_models, X_test, y_test, output_dir='../outputs/figures')

In [ ]:
plot_roc_curves(fitted_models, X_test, y_test, output_dir='../outputs/figures')

In [ ]:
rf_model = fitted_models.get('Random Forest')
if rf_model:
    plot_feature_importance(rf_model, feature_names, top_n=14,
                            output_dir='../outputs/figures')

## 5. Cross-validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                            class_weight='balanced', n_jobs=-1, random_state=42)
cv_results = cross_validate_model(rf, X, y, cv=5)

## 6. Hyperparameter tuning — Random Forest

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test_tune, y_train, y_test_tune = train_test_split(
    X_scaled, y, test_size=0.2, stratify=y, random_state=42
)

best_rf = tune_random_forest(X_train, y_train, cv=3)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_best = best_rf.predict(X_test_tune)
print('Tuned Random Forest — Test Set Report:')
print(classification_report(y_test_tune, y_pred_best,
                             target_names=['No Arrest', 'Arrest']))
print('Confusion Matrix:')
print(confusion_matrix(y_test_tune, y_pred_best))

## 7. Prediction examples

In [ ]:
# Show 10 sample predictions vs ground truth
import joblib

rf_loaded = joblib.load('../outputs/models/Random_Forest.joblib')
sample_idx = np.random.choice(len(X_test), size=10, replace=False)
preds = rf_loaded.predict(X_test[sample_idx])
probs = rf_loaded.predict_proba(X_test[sample_idx])[:, 1]

df_sample = pd.DataFrame({
    'True Label':   y_test[sample_idx],
    'Predicted':    preds,
    'Arrest Prob':  probs.round(3),
    'Correct':      y_test[sample_idx] == preds
})
df_sample